In [1]:
from tinycss2 import tokenizer
%load_ext autoreload
%autoreload 2

# Finding Better Prompts from Retrieved Examples

## Goal

This notebook runs the complete BSc experiment:

```text
Bias-in-Bios → hold out one column → retrieve examples → build prompts
→ score every allowed label as a continuation of each prompt
→ select a prompt on validation rows → evaluate it once on separate test rows
→ inspect predictive quality and group disparities
```

`hard_text` is always input. With target **profession**, gender is the other input and audit group. With target **gender**, profession is the other input and audit group. The structured target value is never included in a validation or test prompt.

Every candidate label is scored against the same master prompt, examples, query, and chat wrapper.

## Setup and assumptions

Use this DataSpell interpreter:

```text
/opt/homebrew/anaconda3/envs/BiasMitigation/bin/python
```

Install `requirements.txt` once. Edit `config.yaml` before running if you want a different target, data size, retrieval method, number/order of examples, master prompt, model, or validation ranking objective.

The first real run can download model files and builds one persistent, model-named LanceDB table per embedding model. Matching later runs reuse those tables instead of embedding the training pool again. Delete `data/lancedb/` before changing data or embedding settings. Semantic conditions compare Qwen3-Embedding-8B and BAAI/bge-large-en-v1.5; training documents remain raw `hard_text`, while each encoder receives its own query prefix. `device: auto` selects CUDA, then Apple MPS, then CPU.

The `label_score` policy computes a mean conditional token log-score for every allowed label and chooses the largest. Scores are relative rankings rather than calibrated probabilities, and inference errors stop the run.

In [2]:
from datasets import load_dataset
from tqdm import tqdm

raw_train = load_dataset(
    "LabHC/bias_in_bios",
    split="train",
)

print(len(raw_train))  # 257478
print(raw_train[0])

257478
{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates', 'profession': 21, 'gender': 0}


In [24]:
from transformers import AutoTokenizer
from pipeline import build_prompt, Column, _apply_chat_template

model_ids = ['Qwen/Qwen3.6-27B', 'Qwen/Qwen3.5-27B', 'Qwen/Qwen3.6-35B-A3B', 'google/gemma-4-31B-it', 'openai/gpt-oss-120b']
# for model_i in model_id:
#     tokenizer = AutoTokenizer.from_pretrained(model_i)
#     print(model_i + ': ', hasattr(tokenizer, 'chat_template'))
# _apply_chat_template([{'role': 'user', 'content': 'hi'}], tokenizer)

In [20]:
tokenizer

TokenizersBackend(name_or_path='openai/gpt-oss-120b', vocab_size=199998, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|return|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	199998: AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	199999: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200000: AddedToken("<|reserved_200000|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200001: AddedToken("<|reserved_200001|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200002: AddedToken("<|return|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200003: AddedToken("<|constrain|>", rstrip=False, lstrip=False, single_word=False, normalized=False, spe

In [25]:
messages = [{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': ''}]
for model_id in model_ids:
    _apply_chat_template(messages, AutoTokenizer.from_pretrained(model_id))

In [ ]:
from pathlib import Path
import sys

import yaml
from IPython.display import Image, display

# DataSpell can start in this folder or in its parent project folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    candidate = PROJECT_ROOT / "fairness-aware-icl"
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError("Could not find config.yaml")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import (  # noqa: E402 - imported after locating the project
    load_config,
    load_data,
    render_input,
    run_experiment,
    task_settings,
    validate_config,
)

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(CONFIG_PATH)
validate_config(config)
DISPLAY_LIMIT = 40
target, audit_column, professions, target_labels = task_settings(config)

condition_count = (
    len(config["retrieval"]["methods"])
    * len(config["retrieval"]["embedding_models"])
    * len(config["retrieval"]["k_values"])
    * len(config["retrieval"]["example_orders"])
    * len(config["prompt_templates"])
)
validation_row_count = (
    len(professions)
    * 2
    * int(config["dataset"]["validation_per_profession_gender"])
)
test_row_count = (
    len(professions)
    * 2
    * int(config["dataset"]["test_per_profession_gender"])
)

print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print(f"Held-out target: {target}")
print(f"Visible input columns: hard_text + {audit_column}")
print(f"Allowed answers: {target_labels}")
print(f"Prompt conditions: {condition_count}")
print(f"Validation rows: {validation_row_count}")
print(f"Final-test rows: {test_row_count}")
print(
    "Prediction operations:",
    condition_count * validation_row_count + test_row_count,
)
print("Each prediction scores every allowed label sequentially.")

## Step 1 — Check the three data partitions and visible fields

Training rows form the retrieval pool. Balanced validation cells select the prompt; disjoint balanced test cells estimate its final performance. The next cell loads the data cache—or downloads missing rows—and shows exactly what may enter a query prompt; it does not load either model.

In [ ]:
train_rows, validation_rows, test_rows_data, dataset_counts = load_data(config, PROJECT_ROOT)

dataset_counts

In [ ]:
'hiii hello my'.title()

In [ ]:
first_rendered_query = render_input(validation_rows[0], target)
print("First model-visible validation query:")
print(first_rendered_query)
print(
    f"True {target} is stored separately for evaluation:",
    validation_rows[0][target],
)

## Step 2 — Select on validation, evaluate once on test

Every prompt condition receives the same validation rows. The configured metric ranks them. Only rank 1 is then inferred on the untouched test rows, avoiding selection on the final evaluation set.

In [ ]:
run = run_experiment(config, PROJECT_ROOT, progress=print)
print("Results folder:", run["run_dir"])
print("Selected prompt file:", run["best_prompt"])
assert set(run["predictions"]["predicted_label"]).issubset(target_labels)
print("Closed-set check passed: every prediction is an allowed label.")

## Step 3 — Compare validation conditions and read the final result

`validation_results` ranks the search conditions. `results` contains one row: the independent final-test metrics for the selected condition. The plot compares validation quality and disparities and annotates that final test score.

In [ ]:
ranking_metric = config["defaults"]["ranking_metric"]
summary_columns = [
    "rank", "selected_for_test", "condition",
    "accuracy", "macro_f1", "balanced_accuracy",
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in summary_columns:
    summary_columns.append(ranking_metric)
print("Validation prompt ranking:")
display(run["validation_results"][summary_columns])

final_columns = [
    "condition", "accuracy", "macro_f1", "balanced_accuracy",
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in final_columns:
    final_columns.append(ranking_metric)
print("Independent final-test result:")
display(run["results"][final_columns])
display(Image(filename=str(run["plot"])))

## Step 4 — Inspect final-test denominators and failure modes

The summary is not enough by itself. Per-class scores show which labels fail, and group rates show the supports behind each disparity. The next cell displays final-test details; all validation details are in the same returned tables with `evaluation_split == "validation"`.

In [ ]:
test_detail_tables = {
    "Per-class metrics": run["class_metrics"].query("evaluation_split == 'test'"),
    "Group disparities": run["fairness_metrics"].query("evaluation_split == 'test'"),
    "Group rates": run["group_metrics"].query("evaluation_split == 'test'"),
    "Confusion counts": run["confusion_matrix"].query("evaluation_split == 'test'"),
}
for table_name, table in test_detail_tables.items():
    print(f"{table_name}: showing {min(len(table), DISPLAY_LIMIT)} of {len(table)} rows")
    display(table.head(DISPLAY_LIMIT))

## Metric and label-scoring guide

For class $c$:

$$Precision_c=\frac{TP_c}{TP_c+FP_c},\quad
Recall_c=\frac{TP_c}{TP_c+FN_c},\quad
F1_c=\frac{2TP_c}{2TP_c+FP_c+FN_c}$$

Accuracy is the fraction of all correct rows. Macro averages give every target class equal weight; weighted averages use class support; balanced accuracy is macro recall. MCC and Cohen's $\kappa$ summarize agreement while accounting for more of the confusion structure.

For `label_score`, allowed class $c$ has continuation tokens $T_c$:

$$score(c)=\frac{1}{|T_c|}\sum_j
\log P(t_j\mid prompt,t_{<j}),\qquad
\hat c=\arg\max_{c\in labels}score(c)$$

Mean normalization reduces the automatic disadvantage of multi-token labels. These are relative ranking scores, not calibrated probabilities. Closed-set choice guarantees a configured label; it does not guarantee correctness or fairness.

For audit group $g$:

$$SR_{c,g}=P(\hat Y=c\mid A=g),\quad
TPR_{c,g}=P(\hat Y=c\mid Y=c,A=g),\quad
FPR_{c,g}=P(\hat Y=c\mid Y\ne c,A=g)$$

The main symmetric disparities are the range across groups: demographic-parity difference uses selection rate, equal-opportunity difference uses TPR, and equalized-odds difference is the larger of the TPR and FPR ranges. Difference metrics are better near 0; demographic-parity ratio is better near 1.

See `README.md` for every formula, undefined-case rule, configuration effect, and interpretation caveat.

## Step 5 — Inspect the selected prompt and its test predictions

`best_prompt.txt` contains the resolved validation-selected master instruction, hyperparameters, validation score, and final test score. Full prompts vary by query because their retrieved examples differ, so they remain in `predictions.csv`.

`label_scores` is a JSON mapping from each allowed label to its mean conditional token log-score.

In [ ]:
print(Path(run["best_prompt"]).read_text(encoding="utf-8"))
selected_condition = run["validation_results"].iloc[0]["condition"]
prediction_columns = [
    "evaluation_split", "query_id", "target", "true_label", "audit_group",
    "predicted_label", "retrieval", "k", "example_order", "prompt_name",
    "label_scores",
]
selected_test_predictions = run["predictions"].query(
    "evaluation_split == 'test' and condition == @selected_condition"
)
print(
    f"Selected test predictions: showing "
    f"{min(len(selected_test_predictions), DISPLAY_LIMIT)} of "
    f"{len(selected_test_predictions)} rows"
)
display(selected_test_predictions[prediction_columns].head(DISPLAY_LIMIT))

## Next run

Change `defaults.target` from `profession` to `gender` and run all cells again. Do not directly compare the two task scores as though they had identical meanings: they have different class sets, and profession is an audit subgroup—not a protected attribute—when gender is the target.

The default one row per validation/test cell is suitable for checking the pipeline but produces coarse disparity metrics. Increase both per-cell settings before drawing thesis conclusions.

The optional Gradio UI exposes the same YAML, guidance, tables, and plot and calls this exact pipeline. Run `app.py` in DataSpell when needed.